# [Module 03 -- Fundamentals] Tasks and Processes

> **MLCourse -- Agentic AI -- CrewAI Fundamentals**

> Tasks are units of work; Processes define how tasks flow. This module covers
> sequential vs hierarchical execution, async kickoff, `kickoff_for_each`,
> callback functions, output parsing, and context passing between tasks.

## What you'll learn

- `Task()` parameters: description, expected_output, agent, context, callback.
- `Process.sequential` vs `Process.hierarchical`.
- `crew.kickoff_async()` -- running crews in async contexts.
- `crew.kickoff_for_each()` -- iterating a crew over a list of inputs.
- Callback functions for post-task processing.
- How task outputs chain together via the `context` parameter.

In [ ]:
# --- Standard library imports -------------------------------------------------
import os
import asyncio
from pathlib import Path

# --- Third-party imports ------------------------------------------------------
from dotenv import load_dotenv

# Walk up to track root.
TRACK = Path.cwd()
while TRACK.name != "03_agentic_ai" and TRACK != TRACK.parent:
    TRACK = TRACK.parent
load_dotenv(TRACK / ".env")

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass

print("Setup complete. Track root:", TRACK)

## 1. Imports

In [ ]:
try:
    from crewai import Agent, Task, Crew, Process
    from langchain_ollama import ChatOllama
    print("[OK] crewai and ChatOllama imported.")
except ImportError as e:
    print("[ERROR] pip install crewai \"crewai[tools]\" langchain-ollama")
    print("        Detail:", e)

In [ ]:
LLM_MODEL = "llama3.1:8b"
llm = ChatOllama(model=LLM_MODEL)

try:
    resp = llm.invoke("Reply OK")
    print("[OK] Ollama live:", resp.content[:30])
except Exception as e:
    print("[WARN] Ollama down. Start it: ollama pull", LLM_MODEL)
    print("       Detail:", e)

## 2. Task parameters in detail

| Parameter          | Type     | Required | Purpose                                        |
|--------------------|----------|----------|------------------------------------------------|
| `description`      | str      | Yes      | Natural-language instruction for the agent.     |
| `expected_output`  | str      | Yes      | Format hint for the agent's response.          |
| `agent`            | Agent    | Yes      | Which agent owns this task.                    |
| `context`          | list     | No       | Prior tasks whose outputs feed into this one.  |
| `callback`         | callable | No       | Function called with the task output on finish.|
| `output_file`      | str      | No       | Path to write the result to disk.              |
| `output_pydantic`  | class    | No       | Pydantic model to parse output into.           |
| `output_json`      | bool     | No       | Parse output as JSON dict.                     |

In [ ]:
# A simple task with all common parameters.
def my_callback(output):
    """Called when the task completes. Receives the CrewOutput object."""
    print("[callback] Task finished. Output length:", len(output.raw))

simple_task = Task(
    description="List three benefits of using AI agents in software engineering.",
    expected_output="A numbered list of exactly 3 items.",
    agent=Agent(
        role="Tech Writer",
        goal="Write clear technical content.",
        backstory="You write concise technical documentation.",
        llm=llm,
        allow_delegation=False,
        verbose=False,
    ),
    callback=my_callback,         # Called after the agent finishes.
    output_file=None,             # Set to a path to write output to disk.
)

print("Task created for:", simple_task.agent.role)
print("Has callback:", simple_task.callback is not None)

## 3. Sequential process

`Process.sequential` runs tasks in the order they appear in the `tasks` list.
Each task can access the outputs of all prior tasks through the `context`
parameter. This is the simplest and most predictable execution mode.

In [ ]:
# Create three agents.
planner = Agent(
    role="Planner",
    goal="Create a brief plan for a blog post.",
    backstory="You are a content strategist.",
    llm=llm,
    allow_delegation=False,
    verbose=False,
)

writer = Agent(
    role="Writer",
    goal="Write the blog post based on the plan.",
    backstory="You are a skilled content writer.",
    llm=llm,
    allow_delegation=False,
    verbose=False,
)

editor = Agent(
    role="Editor",
    goal="Polish the blog post for clarity and grammar.",
    backstory="You are a meticulous editor.",
    llm=llm,
    allow_delegation=False,
    verbose=False,
)

# Create three tasks with context chaining.
plan_task = Task(
    description="Create a 3-point outline for a blog post about Python decorators.",
    expected_output="A numbered outline with 3 main points.",
    agent=planner,
)

write_task = Task(
    description="Write a short blog post (150-200 words) following the outline.",
    expected_output="A complete blog post in markdown format.",
    agent=writer,
    context=[plan_task],          # This task receives plan_task's output.
)

edit_task = Task(
    description="Polish the blog post: fix grammar, improve flow, tighten prose.",
    expected_output="The final edited version of the blog post.",
    agent=editor,
    context=[write_task],         # This task receives write_task's output.
)

# Build the crew with sequential process.
sequential_crew = Crew(
    agents=[planner, writer, editor],
    tasks=[plan_task, write_task, edit_task],
    process=Process.sequential,   # Tasks run in listed order.
    verbose=False,
)

print("Sequential crew:", len(sequential_crew.agents), "agents,", len(sequential_crew.tasks), "tasks")

In [ ]:
try:
    seq_result = sequential_crew.kickoff()
    print("\n=== Final Blog Post ===")
    print(seq_result.raw)
except Exception as e:
    print("[demo skipped]", e)

## 4. Hierarchical process

`Process.hierarchical` adds a **manager agent** that automatically delegates
tasks to the right specialist. CrewAI creates a manager internally (or you
can specify one via the `manager_agent` parameter). The manager decides which
agent handles each task.

> **When to use:** complex workflows where task routing depends on content,
> or when you want a supervisory layer. Adds overhead from extra LLM calls.

In [ ]:
# Define specialist agents.
analyst = Agent(
    role="Market Analyst",
    goal="Analyze market trends with data.",
    backstory="You are a data-driven market analyst.",
    llm=llm,
    allow_delegation=False,
    verbose=False,
)

strategist = Agent(
    role="Strategist",
    goal="Turn analysis into actionable recommendations.",
    backstory="You translate data into strategy.",
    llm=llm,
    allow_delegation=False,
    verbose=False,
)

# Hierarchical crew -- a manager delegates automatically.
hier_task = Task(
    description=(
        "Analyze the rise of edge AI in 2025-2026 and recommend "
        "three strategic actions for a tech startup."
    ),
    expected_output="Market analysis followed by 3 recommendations.",
    agent=analyst,                  # Initial agent; manager may re-delegate.
)

hier_crew = Crew(
    agents=[analyst, strategist],
    tasks=[hier_task],
    process=Process.hierarchical,   # Manager delegates automatically.
    verbose=False,
)

print("Hierarchical crew created with", len(hier_crew.agents), "agents.")

In [ ]:
try:
    hier_result = hier_crew.kickoff()
    print("\n=== Hierarchical Result ===")
    print(hier_result.raw[:500])
except Exception as e:
    print("[demo skipped]", e)

## 5. Context passing between tasks

The `context` parameter on a Task is a list of **other Task objects**. When
the crew runs, the output of each context task is injected into the agent's
prompt. This is how information flows from one task to the next.

Without `context`, each task runs in isolation -- the agent only sees its own
`description`.

In [ ]:
research_task = Task(
    description="Research the top 3 Python web frameworks in 2026.",
    expected_output="A list of 3 frameworks with one-sentence descriptions.",
    agent=planner,
)

summary_task = Task(
    description="Write a comparison table of the 3 frameworks.",
    expected_output="A markdown table with framework name, pros, and cons.",
    agent=writer,
    context=[research_task],       # Writer sees the research output.
)

context_crew = Crew(
    agents=[planner, writer],
    tasks=[research_task, summary_task],
    process=Process.sequential,
    verbose=False,
)

try:
    ctx_result = context_crew.kickoff()
    print("\n=== Context-Passed Result ===")
    print(ctx_result.raw[:500])
except Exception as e:
    print("[demo skipped]", e)

## 6. Callback functions

A callback is a Python function that CrewAI calls the moment a task finishes.
It receives the `CrewOutput` object. Use callbacks for:

- Logging results to a database.
- Triggering downstream pipelines.
- Saving output to disk.
- Validating output before proceeding.

Callbacks run **after** the task completes but **before** the next task starts.

In [ ]:
# Track callback invocations with a mutable list.
callback_log = []

def log_callback(output):
    """Append task info to the log and print a confirmation."""
    callback_log.append({
        "role": output.pydantic.__dict__ if output.pydantic else None,
        "raw_length": len(output.raw),
    })
    print("[callback] Task finished. Raw output length:", len(output.raw))

cb_agent = Agent(
    role="Poet",
    goal="Write a haiku.",
    backstory="You are a master of Japanese poetry.",
    llm=llm,
    allow_delegation=False,
    verbose=False,
)

cb_task = Task(
    description="Write a haiku about machine learning.",
    expected_output="Three lines in 5-7-5 syllable format.",
    agent=cb_agent,
    callback=log_callback,         # This function fires when the task completes.
)

cb_crew = Crew(
    agents=[cb_agent],
    tasks=[cb_task],
    process=Process.sequential,
    verbose=False,
)

try:
    cb_result = cb_crew.kickoff()
    print("\n=== Haiku ===")
    print(cb_result.raw)
    print("\nCallback log:", callback_log)
except Exception as e:
    print("[demo skipped]", e)

## 7. Output parsing -- raw, pydantic, json_dict

`CrewOutput` offers multiple access patterns for the result:

- `.raw` -- plain string from the last task (always available).
- `.pydantic` -- a Pydantic model instance (if `output_pydantic` was set on the task).
- `.json_dict` -- a Python dict (if `output_json=True` was set on the task).
- `.token_usage` -- dict with token counts per provider.

Below we demonstrate `.raw` access and `.token_usage`.

In [ ]:
parse_task = Task(
    description="Name the three primary colors and explain why they are primary.",
    expected_output="A list of 3 colors with one-sentence explanations.",
    agent=Agent(
        role="Art Teacher",
        goal="Explain color theory basics.",
        backstory="You teach foundational art concepts.",
        llm=llm,
        allow_delegation=False,
        verbose=False,
    ),
)

parse_crew = Crew(
    agents=[parse_task.agent],
    tasks=[parse_task],
    process=Process.sequential,
    verbose=False,
)

try:
    parse_result = parse_crew.kickoff()
    print("=== Raw Output ===")
    print(parse_result.raw)
    print("\n=== Token Usage ===")
    for key, val in parse_result.token_usage.items():
        print(f"  {key}: {val}")
except Exception as e:
    print("[demo skipped]", e)

## 8. Async kickoff

`crew.kickoff_async()` returns a coroutine that you can `await` in an async
context. This is useful when you want to run multiple crews concurrently
(e.g. in a FastAPI endpoint or an async Jupyter cell).

> **Note:** CrewAI's async support runs the crew's internal LLM calls in
> threads. The crew itself is still single-threaded -- tasks run sequentially
> within the crew.

In [ ]:
async def run_crew_async():
    """Run a simple crew asynchronously."""
    async_agent = Agent(
        role="Async Responder",
        goal="Reply quickly with a fact.",
        backstory="You are fast.",
        llm=llm,
        allow_delegation=False,
        verbose=False,
    )

    async_task = Task(
        description="Name one fact about the Python programming language.",
        expected_output="A single factual statement.",
        agent=async_agent,
    )

    async_crew = Crew(
        agents=[async_agent],
        tasks=[async_task],
        process=Process.sequential,
        verbose=False,
    )

    result = await async_crew.kickoff_async()   # Non-blocking await.
    return result

try:
    async_result = asyncio.run(run_crew_async())
    print("Async result:", async_result.raw)
except Exception as e:
    print("[demo skipped / async not supported in this context]", e)

## 9. `kickoff_for_each` -- batch execution

`crew.kickoff_for_each(inputs=[...])` runs the same crew multiple times,
once per input dict. Each input is interpolated into task descriptions using
`{key}` placeholders. This is useful for processing a list of items with the
same agent pipeline.

Below we process three Python topics through the same plan-write pipeline.

In [ ]:
batch_planner = Agent(
    role="Batch Planner",
    goal="Create an outline for the given topic.",
    backstory="You plan technical content efficiently.",
    llm=llm,
    allow_delegation=False,
    verbose=False,
)

batch_writer = Agent(
    role="Batch Writer",
    goal="Write a short explanation from the outline.",
    backstory="You write concise technical content.",
    llm=llm,
    allow_delegation=False,
    verbose=False,
)

# Use {topic} placeholder in the description -- kickoff_for_each fills it.
batch_plan = Task(
    description="Create a 2-point outline explaining {topic}.",
    expected_output="A numbered outline with 2 points.",
    agent=batch_planner,
)

batch_write = Task(
    description="Write a 2-3 sentence explanation of {topic} following the outline.",
    expected_output="A short explanation paragraph.",
    agent=batch_writer,
    context=[batch_plan],
)

batch_crew = Crew(
    agents=[batch_planner, batch_writer],
    tasks=[batch_plan, batch_write],
    process=Process.sequential,
    verbose=False,
)

topics = [
    {"topic": "Python decorators"},
    {"topic": "list comprehensions"},
    {"topic": "context managers"},
]

try:
    batch_results = batch_crew.kickoff_for_each(inputs=topics)
    for i, res in enumerate(batch_results):
        print(f"\n=== Topic {i+1}: {topics[i]['topic']} ===")
        print(res.raw[:300])
except Exception as e:
    print("[demo skipped]", e)

## 10. Key takeaways

| Concept                | How                                                |
|------------------------|----------------------------------------------------|
| Sequential process     | `Process.sequential` -- tasks in listed order.     |
| Hierarchical process   | `Process.hierarchical` -- manager auto-delegates.  |
| Context passing        | `context=[prior_task]` on a Task.                  |
| Callbacks              | `callback=my_func` -- fires on task completion.    |
| Async kickoff          | `await crew.kickoff_async()` in async contexts.    |
| Batch execution        | `crew.kickoff_for_each(inputs=[...])`.             |
| Output access          | `.raw`, `.pydantic`, `.json_dict`, `.token_usage`. |

- Tasks are the units of work; Processes are the execution strategy.
- Context chaining lets agents build on each other's outputs.
- Callbacks are lightweight hooks for logging, saving, or validation.
- Next module: built-in tools (file I/O, web scraping, search).